# Library Management System

### Imports

In [1]:
import json
import os
import csv

### Book

In [2]:
class Book:
    def __init__(self, title, author, isbn):
        self.title = title
        self.author = author
        self.isbn = isbn
        self.is_available = True

    def __str__(self):
        status = "Available" if self.is_available else "Checked out"
        return f'"{self.title}" by {self.author} (ISBN: {self.isbn}) - {status}'

### Member, Student, Faculty

In [3]:
class Member:
    def __init__(self, name, member_id):
        self.name = name
        self.member_id = member_id
        self.borrowed_books = []

    def max_books_allowed(self):
        return 0

    def __str__(self):
        return f"{self.name} (ID: {self.member_id}) - {len(self.borrowed_books)}/{self.max_books_allowed()} books borrowed"


class Student(Member):
    def max_books_allowed(self):
        return 3


class Faculty(Member):
    def max_books_allowed(self):
        return 10

### Library
Holds all the books and members, and every action the program can perform.

In [4]:
class Library:
    def __init__(self, data_file="library_data.json"):
        self.data_file = data_file
        self.books = []
        self.members = {}
        self.next_student_id = 1
        self.next_faculty_id = 1
        self.load_data()

    # ---- Books ----

    def add_book(self, title, author, isbn):
        if self.find_book(isbn) is not None:
            print("A book with this ISBN already exists.")
            return
        new_book = Book(title, author, isbn)
        self.books.append(new_book)
        print(f"Added: {new_book}")
        self.save_data()

    def remove_book(self, isbn):
        book = self.find_book(isbn)
        if book is None:
            print("No book found with that ISBN.")
            return
        if not book.is_available:
            print("This book is checked out and can't be removed.")
            return
        self.books.remove(book)
        print(f"Removed: {book.title}")
        self.save_data()

    def search_books(self, keyword):
        keyword = keyword.lower()
        results = []
        for book in self.books:
            if keyword in book.title.lower() or keyword in book.author.lower():
                results.append(book)
        if not results:
            print("No matching books found.")
        else:
            for book in results:
                print(f"  {book}")

    def list_books(self):
        if not self.books:
            print("The library has no books yet.")
            return
        for book in self.books:
            print(f"  {book}")

    def find_book(self, isbn):
        for book in self.books:
            if book.isbn == isbn:
                return book
        return None

    def import_books_from_csv(self, filename, member_id):
        # CSV needs a header row: title,author,isbn — faculty only
        member = self.members.get(member_id)
        if member is None:
            print("No member found with that ID.")
            return
        if not isinstance(member, Faculty):
            print("Only faculty members can bulk-import books.")
            return
        if not os.path.exists(filename):
            print(f"Couldn't find '{filename}'.")
            return

        added = 0
        skipped = 0
        with open(filename, "r") as f:
            reader = csv.DictReader(f)
            for row in reader:
                if self.find_book(row["isbn"]) is not None:
                    skipped += 1
                    continue
                self.books.append(Book(row["title"], row["author"], row["isbn"]))
                added += 1

        self.save_data()
        print(f"Import finished: {added} added, {skipped} skipped.")

    # ---- Members ----

    def register_member(self, name, member_type):
        member_type = member_type.lower()
        if member_type == "student":
            member_id = f"S{self.next_student_id}"
            new_member = Student(name, member_id)
            self.next_student_id += 1
        elif member_type == "faculty":
            member_id = f"F{self.next_faculty_id}"
            new_member = Faculty(name, member_id)
            self.next_faculty_id += 1
        else:
            print("Member type must be 'student' or 'faculty'.")
            return
        self.members[member_id] = new_member
        print(f"Registered: {new_member}")
        self.save_data()

    # ---- Borrowing ----

    def issue_book(self, isbn, member_id):
        book = self.find_book(isbn)
        member = self.members.get(member_id)
        if book is None:
            print("No book found with that ISBN.")
            return
        if member is None:
            print("No member found with that ID.")
            return
        if not book.is_available:
            print("This book is already checked out.")
            return
        if len(member.borrowed_books) >= member.max_books_allowed():
            print(f"{member.name} has reached their borrowing limit.")
            return
        book.is_available = False
        member.borrowed_books.append(isbn)
        print(f"{member.name} borrowed '{book.title}'.")
        self.save_data()

    def return_book(self, isbn, member_id):
        book = self.find_book(isbn)
        member = self.members.get(member_id)
        if book is None or member is None:
            print("Book or member not found.")
            return
        if isbn not in member.borrowed_books:
            print(f"{member.name} doesn't have this book borrowed.")
            return
        book.is_available = True
        member.borrowed_books.remove(isbn)
        print(f"{member.name} returned '{book.title}'.")
        self.save_data()

    def show_member_books(self, member_id):
        member = self.members.get(member_id)
        if member is None:
            print("No member found with that ID.")
            return
        if not member.borrowed_books:
            print(f"{member.name} hasn't borrowed any books.")
            return
        for isbn in member.borrowed_books:
            book = self.find_book(isbn)
            if book:
                print(f"  {book.title}")

    # ---- Save / Load ----

    def save_data(self):
        books_data = []
        for book in self.books:
            books_data.append({
                "title": book.title,
                "author": book.author,
                "isbn": book.isbn,
                "is_available": book.is_available
            })

        members_data = []
        for member in self.members.values():
            member_type = "student" if isinstance(member, Student) else "faculty"
            members_data.append({
                "name": member.name,
                "member_id": member.member_id,
                "type": member_type,
                "borrowed_books": member.borrowed_books
            })

        data = {
            "books": books_data,
            "members": members_data,
            "next_student_id": self.next_student_id,
            "next_faculty_id": self.next_faculty_id
        }
        with open(self.data_file, "w") as f:
            json.dump(data, f, indent=2)

    def load_data(self):
        if not os.path.exists(self.data_file):
            return
        with open(self.data_file, "r") as f:
            data = json.load(f)

        for b in data.get("books", []):
            book = Book(b["title"], b["author"], b["isbn"])
            book.is_available = b["is_available"]
            self.books.append(book)

        for m in data.get("members", []):
            if m["type"] == "student":
                member = Student(m["name"], m["member_id"])
            else:
                member = Faculty(m["name"], m["member_id"])
            member.borrowed_books = m["borrowed_books"]
            self.members[m["member_id"]] = member

        self.next_student_id = data.get("next_student_id", 1)
        self.next_faculty_id = data.get("next_faculty_id", 1)

### Menu

In [5]:
def print_menu():
    print("\n===== Library Menu =====")
    print("1. Add a book")
    print("2. Remove a book")
    print("3. Register a member")
    print("4. Search for a book")
    print("5. List all books")
    print("6. Issue a book")
    print("7. Return a book")
    print("8. View a member's borrowed books")
    print("9. Import many books at once from a CSV file")
    print("10. Exit")

### Main loop

In [6]:
def main():
    library = Library()

    while True:
        print_menu()
        choice = input("Choose an option: ")

        if choice == "1":
            title = input("Book title: ")
            author = input("Author: ")
            isbn = input("ISBN: ")
            library.add_book(title, author, isbn)

        elif choice == "2":
            isbn = input("ISBN of the book to remove: ")
            library.remove_book(isbn)

        elif choice == "3":
            name = input("Member name: ")
            member_type = input("Type (student/faculty): ")
            library.register_member(name, member_type)

        elif choice == "4":
            keyword = input("Search by title or author: ")
            library.search_books(keyword)

        elif choice == "5":
            library.list_books()

        elif choice == "6":
            isbn = input("ISBN of the book to issue: ")
            member_id = input("Member ID: ")
            library.issue_book(isbn, member_id)

        elif choice == "7":
            isbn = input("ISBN of the book to return: ")
            member_id = input("Member ID: ")
            library.return_book(isbn, member_id)

        elif choice == "8":
            member_id = input("Member ID: ")
            library.show_member_books(member_id)

        elif choice == "9":
            member_id = input("Your Member ID (faculty only): ")
            filename = input("CSV filename (e.g. books.csv): ")
            library.import_books_from_csv(filename, member_id)

        elif choice == "10":
            print("Goodbye! Everything's saved.")
            break

        else:
            print("Please enter a number between 1 and 10.")

### Run

In [ ]:
main()


===== Library Menu =====
1. Add a book
2. Remove a book
3. Register a member
4. Search for a book
5. List all books
6. Issue a book
7. Return a book
8. View a member's borrowed books
9. Import many books at once from a CSV file
10. Exit


Choose an option:  3
Member name:  Rahul Tathod
Type (student/faculty):  student


Registered: Rahul Tathod (ID: S1) - 0/3 books borrowed

===== Library Menu =====
1. Add a book
2. Remove a book
3. Register a member
4. Search for a book
5. List all books
6. Issue a book
7. Return a book
8. View a member's borrowed books
9. Import many books at once from a CSV file
10. Exit


Choose an option:  3
Member name:  Anirudh Pathak
Type (student/faculty):  faculty


Registered: Anirudh Pathak (ID: F1) - 0/10 books borrowed

===== Library Menu =====
1. Add a book
2. Remove a book
3. Register a member
4. Search for a book
5. List all books
6. Issue a book
7. Return a book
8. View a member's borrowed books
9. Import many books at once from a CSV file
10. Exit


Choose an option:  9
Your Member ID (faculty only):  F1
CSV filename (e.g. books.csv):  books.csv


Import finished: 15 added, 0 skipped.

===== Library Menu =====
1. Add a book
2. Remove a book
3. Register a member
4. Search for a book
5. List all books
6. Issue a book
7. Return a book
8. View a member's borrowed books
9. Import many books at once from a CSV file
10. Exit
